In [1]:
import pandas as pd
import numpy as np
import random
import warnings
from datetime import datetime



# Suppress all FutureWarnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [2]:
# Backtest trades function
def backtest_trades(price_data, signal_data, tp=None, sl=None, entry_time_offset=None,
                    percentage_change=None, open_order_elimination=None, ignore_time_interval_before=None, ignore_time_interval_after=None):
    output_data = pd.DataFrame(columns=[
        'Datetime', 'Side', 'Signal Open Price', 'Entry Price', 'TP Price', 'SL Price', 'Result', 'Duration','Execution Latency', 'ROI', 'NAV', 'Ignore Reason'
    ])
    
    initial_margin = 100000
    current_margin = initial_margin
    exit_datetimes = []

    
    for i, row in signal_data.iterrows():
        signal_datetime = row['Datetime']
        signal_value = row['Signal']
        
        if signal_value == 0:
            continue
        elif signal_value > 0:
            side = 'Buy'
        else:
            side = 'Sell'
               
    
        adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
        signal_open_price = price_data.at[adjusted_signal_datetime, 'Open']
        
        # Ignoring signals based on open trades
        exit_datetimes.sort(key=lambda x: x[0])
        ignore_signal = False
        reason = ''
        
        if exit_datetimes:
            later_exits = [ed for ed in exit_datetimes if ed[0] > signal_datetime]


            if len(later_exits) >= 2:
                result = 'Ignored'
                reason = ' 2 open trades'
                ignore_signal = True
            elif len(later_exits) == 1:
                if later_exits[-1][1] != side:
                    ignore_signal = False
                else:
                    result = 'Ignored'
                    reason = 'One open trade with the same side'
                    ignore_signal = True


        if ignore_signal:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': result,
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': reason
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        # New logic: ignore signals within a specific time interval before and after events
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            ignore_signal = any(event_datetime - pd.Timedelta(
                minutes=ignore_time_interval_before) <= signal_datetime <= event_datetime + pd.Timedelta(
                minutes=ignore_time_interval_after)
                                for event_datetime in event_times)
        if ignore_signal:
            new_row = pd.DataFrame([{
                    'Datetime': signal_datetime,
                    'Side': side,
                    'Signal Open Price': signal_open_price,
                    'Entry Price': None,
                    'TP Price': None,
                    'SL Price': None,
                    'Result': 'Ignored',
                    'Duration': '00:00:00',
                    'Execution Latency': '00:00:00',
                    'ROI': 0,
                    'NAV': current_margin,
                    'Ignore Reason': 'Signal around economic event'
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue

        entry_datetime, entry_price, entry_duration = determine_entry(price_data, signal_datetime, percentage_change,
                                                                      side, open_order_elimination, entry_time_offset)
        if entry_datetime is None:
            new_row = pd.DataFrame([{
                'Datetime': signal_datetime,
                'Side': side,
                'Signal Open Price': signal_open_price,
                'Entry Price': None,
                'TP Price': None,
                'SL Price': None,
                'Result': 'Not Filled',
                'Duration': '00:00:00',
                'Execution Latency': '00:00:00',
                'ROI': 0,
                'NAV': current_margin,
                'Ignore Reason': ''
            }])
            output_data = pd.concat([output_data, new_row], ignore_index=True)
            continue
        
        # Update NAV on filled order
        current_margin *= (1 - 0.0002)
        
        if side == 'Buy':
            tp_price = entry_price * (1 + tp)
            sl_price = entry_price * (1 - sl)
        else:
            tp_price = entry_price * (1 - tp)
            sl_price = entry_price * (1 + sl)
        
        result, duration_str = check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side)
        exit_datetime = entry_datetime + pd.Timedelta(duration_str)

        if result in [1, -1]:
            exit_datetimes.append((exit_datetime, side))
        
        if 'event_datetimes' in row and pd.notna(row['event_datetimes']):
            event_times = [pd.to_datetime(e.strip()) for e in str(row['event_datetimes']).split(',')]
            for event_datetime in event_times:
                if entry_datetime < event_datetime < exit_datetime:
                    exit_datetime = event_datetime - pd.Timedelta(minutes=10)
                    if exit_datetime in price_data.index:
                        exit_price = price_data.at[exit_datetime, 'Open']
                        result = 'ended before data'
                        if (side == 'Buy' and exit_price > entry_price) or (side == 'Sell' and exit_price < entry_price):
                            result += ' with profit'
                            pct_change = (exit_price - entry_price) / entry_price if side == 'Buy' else (entry_price - exit_price) / entry_price
                            current_margin = current_margin * (1 + pct_change)
                        else:
                            result += ' with loss'
                            pct_change = (entry_price - exit_price) / entry_price if side == 'Buy' else (exit_price - entry_price) / entry_price
                            current_margin = current_margin * (1 - pct_change)
                    break

        if result not in ['ended before data with profit', 'ended before data with loss', 'ended before data with no exact price']:
            
            if result == 1:
                current_margin = current_margin * (1 + tp)
                current_margin *= (1 - 0.0005)
            elif result == -1:
                current_margin = current_margin * (1 - sl)
                current_margin *= (1 - 0.0005)

        
        roi = ((current_margin - initial_margin) / initial_margin) * 100
        nav = current_margin
        initial_margin = current_margin
        
        new_row = pd.DataFrame([{
            'Datetime': signal_datetime,
            'Side': side,
            'Signal Open Price': signal_open_price,
            'Entry Price': entry_price,
            'TP Price': tp_price,
            'SL Price': sl_price,
            'Result': result,
            'Duration': duration_str,
            'Execution Latency': format_duration(entry_duration),
            'ROI': roi,
            'NAV': nav,
            'Ignore Reason': ''
        }])
        
        output_data = pd.concat([output_data, new_row], ignore_index=True)
     
    # Ensure 'Datetime' column is in datetime format
    output_data['Datetime'] = pd.to_datetime(output_data['Datetime'])
    
        
        # Calculate Daily Return
    output_data['Date'] = output_data['Datetime'].dt.date
    daily_nav = output_data.groupby('Date')['NAV'].last().to_dict()
    daily_returns = {}
    previous_day_nav = 100000
    
    for date, nav in daily_nav.items():
        daily_return = ((nav - previous_day_nav) / previous_day_nav) * 100
        daily_returns[date] = daily_return
        previous_day_nav = nav

    output_data['Daily Return'] = output_data['Date'].map(daily_returns)
    output_data.drop(columns=['Date'], inplace=True)    
    
    return output_data


# Helper functions
def format_duration(duration):
    seconds = duration.total_seconds()
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    seconds = int(seconds % 60)
    return f"{hours:02}:{minutes:02}:{seconds:02}"

def check_tp_sl(price_data, entry_datetime, tp_price, sl_price, side):
    result = 0
    exit_datetime = None
    subsequent_prices = price_data.loc[entry_datetime:]

    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy':
            if price_row['High'] >= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['Low'] <= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
        else:
            if price_row['Low'] <= tp_price:
                result = 1
                exit_datetime = current_datetime
                break
            elif price_row['High'] >= sl_price:
                result = -1
                exit_datetime = current_datetime
                break
    
    if exit_datetime:
        duration = exit_datetime - entry_datetime
        duration_str = format_duration(duration)
    else:
        duration_str = '00:00:00'
        
    return result, duration_str

def determine_entry(price_data, signal_datetime, percentage_change, side, open_order_elimination, entry_time_offset):
    adjusted_signal_datetime = signal_datetime + pd.Timedelta(minutes=entry_time_offset)
    if adjusted_signal_datetime not in price_data.index:
        return None, None, None
    
    adjusted_open_price = price_data.at[adjusted_signal_datetime, 'Open']
    percentage_change_price = adjusted_open_price * (1 - percentage_change) if side == 'Buy' else adjusted_open_price * (1 + percentage_change)
    
    time_limit = adjusted_signal_datetime + pd.Timedelta(minutes=open_order_elimination)
    subsequent_prices = price_data.loc[adjusted_signal_datetime:time_limit]
    
    for current_datetime, price_row in subsequent_prices.iterrows():
        if side == 'Buy' and price_row['Low'] <= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration
        elif side == 'Sell' and price_row['High'] >= percentage_change_price:
            entry_price = percentage_change_price
            duration = current_datetime - adjusted_signal_datetime
            return current_datetime, entry_price, duration

    return None, None, None

In [3]:
price_data = pd.read_csv('E:\Signal Backtesting\Input\price_2023-06-01_to_2024-08-4_min.csv', parse_dates=['Datetime'],
                         index_col='Datetime')
# Re-load the signal data without setting the index
signal_data = pd.read_csv('E:\Signal Backtesting\Input\\filtered_signals_with_all_events_plus_one_hour.csv', parse_dates=['Datetime'])
# month = 7  # January (you can change this to the desired month)
# year = 2024  # You can change this to the desired year
# # 
# # # Filter the signal data for the specified month and year
# signal_data = signal_data[(signal_data['Datetime'].dt.year == year)]

In [ ]:
# Define the parameters
tp = 0.01
sl = 0.01
entry_time_offset = 0 # Time offset in minutes
percentage_change = 0
open_order_elimination = 0 
# Call the backtest_trades function
backtest_output = backtest_trades(
    price_data=price_data,
    signal_data=signal_data,
    tp=tp,
    sl=sl,
    entry_time_offset=entry_time_offset,
    percentage_change=percentage_change,
    open_order_elimination=open_order_elimination,
    ignore_time_interval_before=720,
    ignore_time_interval_after=0
)
 
backtest_output

In [4]:
import numpy as np
import random

# Define the parameter ranges
tp_values = np.arange(0.009, 0.014, 0.001)
sl_values = np.arange(0.009, 0.014, 0.001)
entry_time_offset_values = np.arange(0, 120, 10)
percentage_change_values = np.arange(0.0001, 0.0015, 0.0001)
ignore_time_interval_before= np.arange(0,1080,60)

# Evaluation function
def evaluate(individual):
    tp, sl, entry_time_offset, percentage_change,ignore_time_interval_before = individual
    monthly_rois = []
    penalty_weight = 2  # Penalty weight for negative ROIs
    variance_weight = 1  # Weight for the variance component

    # Iterate over each month from January to July
    for month in range(1, 8):
        # Filter the data by both year and month
        month_signal_data = signal_data[
            (signal_data['Datetime'].dt.year == specific_year) &
            (signal_data['Datetime'].dt.month == month)
        ]

        # Run backtest with given parameters
        result = backtest_trades(
            price_data, month_signal_data, tp=tp, sl=sl, 
            entry_time_offset=entry_time_offset, 
            percentage_change=percentage_change, 
            open_order_elimination=120, 
            ignore_time_interval_before=ignore_time_interval_before, 
            ignore_time_interval_after=0
        )

        # Calculate the final NAV
        final_nav = result['NAV'].iloc[-1]

        # Calculate ROI
        roi = ((final_nav - 100000) / 100000) * 100
        monthly_rois.append(roi)

        # Print the ROI for the current month
        print(f"Individual {individual}: Month {month} ROI: {roi:.2f}%")

    # Calculate the sum of ROIs
    sum_roi = sum(monthly_rois)

    # Apply a penalty for any negative ROIs
    penalty = sum(penalty_weight * roi for roi in monthly_rois if roi < 0)

    # Calculate the variance of ROIs
    variance_roi = np.var(monthly_rois)

    # Objective: Maximize the sum of ROIs while minimizing the penalty and variance
    fitness = sum_roi + penalty - variance_weight * variance_roi

    print(f"Sum of ROIs: {sum_roi:.2f}, Penalty: {penalty:.2f}, Variance: {variance_roi:.4f}, Fitness: {fitness:.4f}")
    return fitness

# Randomly initialize an individual
def create_individual():
    while True:
        tp = np.random.choice(tp_values)
        sl = np.random.choice(sl_values)
        if sl < tp:  # Ensure that SL is less than TP
            break
    return [
        tp,
        sl,
        np.random.choice(entry_time_offset_values),
        np.random.choice(percentage_change_values),
        np.random.choice(ignore_time_interval_before)
    ]

# Mutate an individual
def mutate(individual):
    while True:
        index = random.randint(0, len(individual) - 1)
        if index == 0:  # Mutate TP
            tp = np.random.choice(tp_values)
            if individual[1] < tp:  # Ensure SL < TP
                individual[index] = tp
                break
        elif index == 1:  # Mutate SL
            sl = np.random.choice(sl_values)
            if sl < individual[0]:  # Ensure SL < TP
                individual[index] = sl
                break
        elif index == 2:
            individual[index] = np.random.choice(entry_time_offset_values)
            break
        elif index == 3:
            individual[index] = np.random.choice(percentage_change_values)
            break
        elif index == 4:
            individual[index] = np.random.choice(ignore_time_interval_before)
            break    
    return individual

# Simulated Annealing algorithm
def simulated_annealing():
    current_individual = create_individual()
    current_fitness = evaluate(current_individual)
    best_individual = list(current_individual)
    best_fitness = current_fitness
    
    initial_temperature = 1.0
    final_temperature = 0.01
    alpha = 0.99
    temperature = initial_temperature
    
    step = 0
    while temperature > final_temperature:
        step += 1
        new_individual = mutate(list(current_individual))
        new_fitness = evaluate(new_individual)
        
        print(f"Step {step}:")
        print(f"  Current Individual: {current_individual}")
        print(f"  Current Fitness: {current_fitness}")
        print(f"  New Individual: {new_individual}")
        print(f"  New Fitness: {new_fitness}")
        print(f"  Temperature: {temperature:.4f}")
        
        
        if new_fitness > current_fitness or random.uniform(0, 1) < np.exp((new_fitness - current_fitness) / temperature):
            current_individual = new_individual
            current_fitness = new_fitness
        
        if current_fitness > best_fitness:
            best_individual = list(current_individual)
            best_fitness = current_fitness
        
        temperature *= alpha
    
    best_tp, best_sl, best_entry_time_offset, best_percentage_change, best_ignore_time_before = best_individual
    optimized_fitness = best_fitness
    
    print(f"\nFinal Optimized Results for Year {specific_year}:")
    print(f"  Best Take Profit: {best_tp}")
    print(f"  Best Stop Loss: {best_sl}")
    print(f"  Best Entry Time Offset: {best_entry_time_offset}")
    print(f"  Best Percentage Change: {best_percentage_change}")
    print(f"  Best Ignore Time: {best_ignore_time_before}")
    print(f"  Optimized Fitness (Sum ROI - Variance): {optimized_fitness:.4f}")


def optimize_for_jan_to_july_2024():
    global specific_year
    specific_year = 2024
    simulated_annealing()

# Run the optimization for January to July 2024
optimize_for_jan_to_july_2024()

Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 1 ROI: -9.76%
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 2 ROI: -5.62%
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 3 ROI: -11.19%
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 4 ROI: -0.35%
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 5 ROI: -0.53%
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 6 ROI: -4.66%
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0009000000000000001, 240]: Month 7 ROI: 4.96%
Sum of ROIs: -27.16, Penalty: -64.24, Variance: 27.6611, Fitness: -119.0613
Individual [0.011999999999999997, 0.009999999999999998, 30, 0.0013000000000000002, 240]: Month 1 ROI: -8.79%
Individual [0.011999999999999997, 0.0099999999999999

KeyboardInterrupt: 

In [ ]:
import os
import pandas as pd
import logging

def calculate_metrics(group, initial_nav):
    total_trades = len(group[(group['Result'] == 1) | (group['Result'] == -1)])
    total_wins = len(group[group['Result'] == 1])
    total_losses = len(group[group['Result'] == -1])
    win_rate = total_wins / total_trades if total_trades > 0 else 0
    
    final_nav = group['NAV'].iloc[-1] if total_trades > 0 else initial_nav
    roi = ((final_nav - initial_nav) / initial_nav) * 100
    
    return {
        'Total Trades': total_trades,
        'Total Wins': total_wins,
        'Total Losses': total_losses,
        'Win Rate': win_rate,
        'ROI': roi,
        'NAV': final_nav
    }

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

def generate_report(price_data, signal_data, tp_sl_dict, output_directory):
    report_columns = ['Date', 'TP', 'SL', 'Total Trades', 'Total Wins', 'Total Losses', 'Win Rate', 'ROI', 'NAV']
    
    # Ensure the output directory exists
    os.makedirs(output_directory, exist_ok=True)

    tp_values = next(iter(tp_sl_dict.values()))['tp']
    sl_values = next(iter(tp_sl_dict.values()))['sl']

    for tp, sl in zip(tp_values, sl_values):
        logging.info(f'Starting TP/SL combination: TP={tp}, SL={sl}')
        report_data = pd.DataFrame(columns=report_columns)
        

        for period, params in tp_sl_dict.items():
            logging.info(f'Processing period: {period}')
            period_data = signal_data[signal_data['Datetime'].dt.to_period('M') == period]

            # Backtest the trades for the current scenario
            trade_data = backtest_trades(price_data, period_data, tp, sl, entry_time_offset=0, 
                                         percentage_change=0, open_order_elimination=0, ignore_time_interval_before=720, 
                                         ignore_time_interval_after=0)
            
            monthly_groups = trade_data.groupby(trade_data['Datetime'].dt.to_period('M'))
            monthly_reports = []
            initial_nav = 100000
            for month, group in monthly_groups:
                logging.info(f'Calculating metrics for month: {month}')
                
                metrics = calculate_metrics(group, initial_nav)
                metrics['Date'] = month.strftime('%Y-%m')
                metrics['TP'] = tp
                metrics['SL'] = sl
                monthly_reports.append(pd.DataFrame([metrics]))
                initial_nav = metrics['NAV']

            if monthly_reports:
                monthly_report = pd.concat(monthly_reports, ignore_index=True)
                report_data = pd.concat([report_data, monthly_report], ignore_index=True)
        now = datetime.now()        
        formatted_date = now.strftime('%Y-%m-%d')
        output_file_name = f'report_TP{tp}_SL{sl}_with_12H_ignore_signal_before_most_important_events_{formatted_date}.csv'
        report_data.to_csv(os.path.join(output_directory, output_file_name), index=False)
        logging.info(f'Report saved: {output_file_name}')

    logging.info('Report generation complete')
    return


In [11]:
# Define tp and sl values for each month
tp_sl_dict = {
    pd.Period('2024-01', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], },
    pd.Period('2024-02', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014]},
    pd.Period('2024-03', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014]},
    pd.Period('2024-04', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014]},
    pd.Period('2024-05', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014]},
    pd.Period('2024-06', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014]},
    pd.Period('2024-07', 'M'): {'tp': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014], 'sl': [0.009, 0.01, 0.011, 0.012, 0.013, 0.014]},
    # Add more periods with corresponding tp and sl values as needed
}

output_directory = 'E:\Signal Backtesting\Output\Backtest\monthly_backtest_Jan_to_July_2024_with_tp=sl_report'

# Generate the report
report_data = generate_report(price_data, signal_data, tp_sl_dict, output_directory)

2024-08-10 16:53:48,448 - INFO - Starting report generation for TP=0.012 and SL=0.011
2024-08-10 16:53:48,450 - INFO - Processing period: 2024-01
2024-08-10 16:53:49,662 - INFO - Calculating metrics for month: 2024-01
2024-08-10 16:53:49,665 - INFO - Processing period: 2024-02
2024-08-10 16:53:50,171 - INFO - Calculating metrics for month: 2024-02
2024-08-10 16:53:50,173 - INFO - Processing period: 2024-03
2024-08-10 16:53:50,916 - INFO - Calculating metrics for month: 2024-03
2024-08-10 16:53:50,918 - INFO - Processing period: 2024-04
2024-08-10 16:53:51,532 - INFO - Calculating metrics for month: 2024-04
2024-08-10 16:53:51,535 - INFO - Processing period: 2024-05
2024-08-10 16:53:52,293 - INFO - Calculating metrics for month: 2024-05
2024-08-10 16:53:52,295 - INFO - Processing period: 2024-06
2024-08-10 16:53:53,190 - INFO - Calculating metrics for month: 2024-06
2024-08-10 16:53:53,192 - INFO - Processing period: 2024-07
2024-08-10 16:53:53,775 - INFO - Calculating metrics for month